# Setup

Reads both sheets and creates a combined version

In [ ]:
import pandas as pd
import re

file1 = pd.read_csv("Physical Merch Collection Lists - Sheet13.csv",skiprows=range(1,3))
file2 = pd.read_csv("Physical Merch Collection Lists - Sheet14.csv",skiprows=range(1,3))

combined = pd.concat([file1, file2], ignore_index=True)

Establishes all Pokemon to scan for and checks for the patterns

In [2]:
base_names = [
    "Cleffa", "Clefairy", "Clefable",
    "Lotad", "Lombre", "Ludicolo",
    "Piplup", "Prinplup", "Empoleon",
    "Venipede", "Whirlipede", "Scolipede",
    "Fletchling", "Fletchinder", "Talonflame",
    "Litten", "Torracat", "Incineroar",
    "Mudbray", "Mudsdale",
    "Farfetch'd", "Sirfetch'd",
    "Sneasler", "Sneasel",
    "Rookidee", "Corvisquire", "Corviknight",
    "Chien-Pao"
]

patterns = {
    name: re.compile(rf"\b{re.escape(name)}\b", re.IGNORECASE)
    for name in sorted(base_names, key=len, reverse=True)
}

Pre-processes by setting up needed categories and cleans up pricing, Grandmaster status (yes/no) and base name

In [3]:
def preprocess(df):
    df = df.copy()

    df = df[
        [
            "Card Name",
            "Set/Type Name",
            "TCGPlayer Category",
            "Grandmaster?",
            "Price"
        ]
    ]

    # Clean price
    df["Price"] = (
        df["Price"]
        .astype(str)
        .str.replace(r"[^\d.]", "", regex=True)
    )
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce").fillna(0)

    # Normalize Grandmaster
    df["Grandmaster?"] = df["Grandmaster?"].str.strip().str.lower()

    # Base name extraction
    def extract_base_name(card_name):
        card_name = str(card_name)

        for base, pattern in patterns.items():  # <-- now defined globally
            if pattern.search(card_name):
                return base

        return "Other"

    df["Base Name"] = df["Card Name"].apply(extract_base_name)

    return df

Preprocesses all 3 files

In [4]:
df1 = preprocess(file1)
df2 = preprocess(file2)
df_combined = preprocess(combined)

# Gens 7-9 (Sheet 13)

Shows the total price and count for each Pokemon name, sorted by price

In [5]:
df1.groupby("Base Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Base Name,,
Piplup,8,627.51
Clefairy,3,317.00
Cleffa,2,309.84
Chien-Pao,3,186.99
Corviknight,4,109.84
Clefable,1,70.00
Incineroar,6,57.17
Mudsdale,2,41.81
Sneasler,3,40.99


Shows the total price and count for each Pokemon name, sorted by price. Excludes any variations not found in the main sets

In [6]:
df1[df1["Grandmaster?"] == "no"].groupby("Base Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Base Name,,
Piplup,5,526.72
Clefairy,2,310.00
Clefable,1,70.00
Incineroar,2,31.21
Scolipede,1,27.00
Sneasler,2,25.99
Whirlipede,1,16.00
Chien-Pao,1,7.00
Corviknight,1,5.85


Shows the total price and count for each Pokemon name, sorted by price. Excludes any main set cards

In [7]:
df1[df1["Grandmaster?"] == "yes"].groupby("Base Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Base Name,,
Cleffa,2,309.84
Chien-Pao,2,179.99
Corviknight,3,103.99
Piplup,3,100.79
Mudsdale,2,41.81
Empoleon,1,28.00
Incineroar,4,25.96
Sneasler,1,15.00
Clefairy,1,7.00


Shows the total price and count for each set, sorted by price. 

In [8]:
df1.groupby("Set/Type Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Set/Type Name,,
Sun & Moon - Cosmic Eclipse,6,740.57
Mega Evolution - Ascended Heroes,1,190.00
SVP Black Star Promos,3,187.98
Scarlet & Violet - Journey Together,2,127.00
Sword & Shield - Battle Styles,4,115.99
Sun & Moon - Ultra Prism,2,99.99
Scarlet & Violet - Paldea Evolved,1,95.00
Mega Evolution - Perfect Order,1,70.00
SM Black Star Promos,3,49.05


In [9]:
df1[df1["Grandmaster?"] == "no"].groupby("Set/Type Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Set/Type Name,,
Sun & Moon - Cosmic Eclipse,5,526.72
Mega Evolution - Ascended Heroes,1,190.00
Scarlet & Violet - Journey Together,1,120.00
Mega Evolution - Perfect Order,1,70.00
Scarlet & Violet - Black Bolt,2,43.00
Sword & Shield - Astral Radiance,2,25.99
Sun & Moon - Guardians Rising,1,25.69
SVP Black Star Promos,1,7.00
Sword & Shield - Silver Tempest,1,5.85


In [10]:
df1[df1["Grandmaster?"] == "yes"].groupby("Set/Type Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Set/Type Name,,
Sun & Moon - Cosmic Eclipse,1,213.85
SVP Black Star Promos,2,180.98
Sword & Shield - Battle Styles,3,111.99
Sun & Moon - Ultra Prism,2,99.99
Scarlet & Violet - Paldea Evolved,1,95.00
SM Black Star Promos,2,48.55
Sword & Shield - Darkness Ablaze,1,20.00
Sword & Shield - Lost Origin,1,15.00
Sun & Moon,2,14.50


Shows the total price and count for each TCGPlayer Category, sorted by price.

In [11]:
df1.groupby("TCGPlayer Category").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
TCGPlayer Category,,
SM - Cosmic Eclipse,5,526.72
Alternate Art Promos,1,213.85
ME: Ascended Heroes,1,190.00
SV: Scarlet & Violet Promo Cards,3,187.98
Prize Pack Series Cards,6,136.84
SV09: Journey Together,1,120.00
Jumbo Cards,5,115.95
Miscellaneous Cards & Products,3,91.99
ME03: Perfect Order,1,70.00


In [12]:
df1[df1["Grandmaster?"] == "no"].groupby("TCGPlayer Category").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
TCGPlayer Category,,
SM - Cosmic Eclipse,5,526.72
ME: Ascended Heroes,1,190.00
SV09: Journey Together,1,120.00
ME03: Perfect Order,1,70.00
SV: Black Bolt,2,43.00
SWSH10: Astral Radiance,2,25.99
SM - Guardians Rising,1,25.69
SV: Scarlet & Violet Promo Cards,1,7.00
SWSH12: Silver Tempest Trainer Gallery,1,5.85


In [13]:
df1[df1["Grandmaster?"] == "yes"].groupby("TCGPlayer Category").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
TCGPlayer Category,,
Alternate Art Promos,1,213.85
SV: Scarlet & Violet Promo Cards,2,180.98
Prize Pack Series Cards,6,136.84
Jumbo Cards,5,115.95
Miscellaneous Cards & Products,3,91.99
SM Promos,1,39.90
SM Trainer Kit: Alolan Sandslash & Alolan Ninetales,1,35.00
First Partner Pack,2,2.15
Deck Exclusives,1,1.91


Shows the total number of cards and total pricing

In [14]:
df1.agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Price
Count,43.00
Total_Price,1845.08


Shows the total number of cards and total pricing, only including cards found outside of main sets

In [15]:
df1[df1["Grandmaster?"] == "yes"].agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Price
Count,22.00
Total_Price,818.57


Shows the total number of cards and total pricing, only including cards found within the main sets.

In [16]:
df1[df1["Grandmaster?"] == "no"].agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Price
Count,21.00
Total_Price,1026.51


Sorts cards by pricing "buckets" (Cheap, Moderately Priced, Kinda Expensive, Ridiculously Expensive). Shows the total count and price for each bucket.

In [17]:
bins = [0, 5, 20, 50, float("inf")]
labels = ["0-5", "5.01-20", "20.01-49.99", "50+"]

df1["Price Bucket"] = pd.cut(
    df1["Price"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df1.groupby(["Grandmaster?", "Price Bucket"], observed=False).agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

Count  Total_Price
Grandmaster? Price Bucket                    
no           0-5               6         9.74
             5.01-20           4        34.37
             20.01-49.99       4       120.67
             50+               7       861.73
yes          0-5               6        11.71
             5.01-20           7        84.14
             20.01-49.99       3       102.90
             50+               6       619.82

In [18]:
bins = [0, 5, 20, 50, float("inf")]
labels = ["0-5", "5.01-20", "20.01-49.99", "50+"]

df1["Price Bucket"] = pd.cut(
    df1["Price"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df1.groupby(["Price Bucket"], observed=False).agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Count,Total_Price
Price Bucket,,
0-5,12,21.45
5.01-20,11,118.51
20.01-49.99,7,223.57
50+,13,1481.55


# Gens 1-6

In [19]:
df2.groupby("Base Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Base Name,,
Clefable,11,1308.17
Clefairy,10,451.94
Empoleon,8,245.35
Cleffa,13,213.02
Piplup,12,187.36
Ludicolo,6,180.96
Lotad,6,103.70
Lombre,6,69.39
Scolipede,2,28.99


In [20]:
df2[df2["Grandmaster?"] == "no"].groupby("Base Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Base Name,,
Empoleon,5,237.86
Cleffa,9,184.78
Clefairy,6,173.67
Ludicolo,5,172.96
Clefable,7,164.25
Piplup,4,114.94
Lotad,4,77.70
Lombre,5,49.40
Scolipede,1,8.99


In [21]:
df2[df2["Grandmaster?"] == "yes"].groupby("Base Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Base Name,,
Clefable,4,1143.92
Clefairy,4,278.27
Piplup,8,72.42
Cleffa,4,28.24
Lotad,2,26.00
Scolipede,1,20.00
Lombre,1,19.99
Ludicolo,1,8.00
Empoleon,3,7.49


In [22]:
df2.groupby("Set/Type Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Set/Type Name,,
Jungle,3,1075.17
Expedition Base Set,4,265.76
Base Set,1,239.99
Black & White - Plasma Freeze,1,180.00
EX Deoxys,7,137.23
Diamond & Pearl - Majestic Dawn,6,110.86
EX Sandstorm,6,109.88
EX Unseen Forces,3,103.14
Gym Heroes,2,93.74


In [23]:
df2[df2["Grandmaster?"] == "no"].groupby("Set/Type Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Set/Type Name,,
Expedition Base Set,4,265.76
Black & White - Plasma Freeze,1,180.00
EX Deoxys,5,107.23
EX Unseen Forces,3,103.14
EX Sandstorm,4,85.89
Skyridge,2,81.96
Platinum,2,79.70
Diamond & Pearl - Majestic Dawn,1,64.99
Diamond & Pearl,5,56.87


In [24]:
df2[df2["Grandmaster?"] == "yes"].groupby("Set/Type Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Set/Type Name,,
Jungle,3,1075.17
Base Set,1,239.99
Gym Heroes,2,93.74
Diamond & Pearl - Majestic Dawn,5,45.87
EX Deoxys,2,30.00
EX Sandstorm,2,23.99
Black & White,1,20.00
Neo Genesis,2,18.75
DP Black Star Promos,1,17.30


In [25]:
df2.groupby("TCGPlayer Category").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
TCGPlayer Category,,
WoTC Promos,1,998.20
Expedition,4,265.76
Base Set (Shadowless),1,239.99
Plasma Freeze,1,180.00
World Championship Decks,16,118.75
Deoxys,5,107.23
Unseen Forces,3,103.14
Gym Heroes,2,93.74
Sandstorm,4,85.89


In [26]:
df2[df2["Grandmaster?"] == "no"].groupby("TCGPlayer Category").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
TCGPlayer Category,,
Expedition,4,265.76
Plasma Freeze,1,180.00
Deoxys,5,107.23
Unseen Forces,3,103.14
Sandstorm,4,85.89
Skyridge,2,81.96
Platinum,2,79.70
Majestic Dawn,1,64.99
Diamond and Pearl,5,56.87


In [27]:
df2[df2["Grandmaster?"] == "yes"].groupby("TCGPlayer Category").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
TCGPlayer Category,,
WoTC Promos,1,998.20
Base Set (Shadowless),1,239.99
World Championship Decks,16,118.75
Gym Heroes,2,93.74
Jungle,2,76.97
Burger King Promos,1,28.80
Blister Exclusives,2,21.02
Neo Genesis,2,18.75
Jumbo Cards,1,5.00


In [28]:
df2.agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Price
Count,83.00
Total_Price,2808.35


In [29]:
df2[df2["Grandmaster?"] == "yes"].agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Price
Count,32.00
Total_Price,1612.51


In [30]:
df2[df2["Grandmaster?"] == "no"].agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Price
Count,51.00
Total_Price,1195.84


In [31]:
bins = [0, 5, 20, 50, float("inf")]
labels = ["0-5", "5.01-20", "20.01-49.99", "50+"]

df2["Price Bucket"] = pd.cut(
    df2["Price"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df2.groupby(["Grandmaster?", "Price Bucket"], observed=False).agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

Count  Total_Price
Grandmaster? Price Bucket                    
no           0-5              17        51.41
             5.01-20          16       154.18
             20.01-49.99      11       352.85
             50+               7       637.40
yes          0-5              15        33.42
             5.01-20          10       127.39
             20.01-49.99       3        75.79
             50+               4      1375.91

In [32]:
bins = [0, 5, 20, 50, float("inf")]
labels = ["0-5", "5.01-20", "20.01-49.99", "50+"]

df2["Price Bucket"] = pd.cut(
    df2["Price"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df2.groupby(["Price Bucket"], observed=False).agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Count,Total_Price
Price Bucket,,
0-5,32,84.83
5.01-20,26,281.57
20.01-49.99,14,428.64
50+,11,2013.31


# Everything

In [33]:
df_combined.groupby("Base Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Base Name,,
Clefable,12,1378.17
Piplup,20,814.87
Clefairy,13,768.94
Cleffa,15,522.86
Empoleon,10,277.35
Chien-Pao,3,186.99
Ludicolo,7,183.81
Corviknight,4,109.84
Lotad,6,103.70


In [34]:
df_combined[df_combined["Grandmaster?"] == "no"].groupby("Base Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Base Name,,
Piplup,9,641.66
Clefairy,8,483.67
Empoleon,6,241.86
Clefable,8,234.25
Cleffa,9,184.78
Ludicolo,5,172.96
Lotad,4,77.70
Lombre,5,49.40
Scolipede,2,35.99


In [35]:
df_combined[df_combined["Grandmaster?"] == "yes"].groupby("Base Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Base Name,,
Clefable,4,1143.92
Cleffa,6,338.08
Clefairy,5,285.27
Chien-Pao,2,179.99
Piplup,11,173.21
Corviknight,3,103.99
Mudsdale,2,41.81
Empoleon,4,35.49
Lotad,2,26.00


In [36]:
df_combined.groupby("Set/Type Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Set/Type Name,,
Jungle,3,1075.17
Sun & Moon - Cosmic Eclipse,6,740.57
Expedition Base Set,4,265.76
Base Set,1,239.99
Mega Evolution - Ascended Heroes,1,190.00
SVP Black Star Promos,3,187.98
Black & White - Plasma Freeze,1,180.00
EX Deoxys,7,137.23
Scarlet & Violet - Journey Together,2,127.00


In [37]:
df_combined[df_combined["Grandmaster?"] == "no"].groupby("Set/Type Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Set/Type Name,,
Sun & Moon - Cosmic Eclipse,5,526.72
Expedition Base Set,4,265.76
Mega Evolution - Ascended Heroes,1,190.00
Black & White - Plasma Freeze,1,180.00
Scarlet & Violet - Journey Together,1,120.00
EX Deoxys,5,107.23
EX Unseen Forces,3,103.14
EX Sandstorm,4,85.89
Skyridge,2,81.96


In [38]:
df_combined[df_combined["Grandmaster?"] == "yes"].groupby("Set/Type Name").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
Set/Type Name,,
Jungle,3,1075.17
Base Set,1,239.99
Sun & Moon - Cosmic Eclipse,1,213.85
SVP Black Star Promos,2,180.98
Sword & Shield - Battle Styles,3,111.99
Sun & Moon - Ultra Prism,2,99.99
Scarlet & Violet - Paldea Evolved,1,95.00
Gym Heroes,2,93.74
SM Black Star Promos,2,48.55


In [39]:
df_combined.groupby("TCGPlayer Category").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
TCGPlayer Category,,
WoTC Promos,1,998.20
SM - Cosmic Eclipse,5,526.72
Expedition,4,265.76
Base Set (Shadowless),1,239.99
Alternate Art Promos,1,213.85
ME: Ascended Heroes,1,190.00
SV: Scarlet & Violet Promo Cards,3,187.98
Plasma Freeze,1,180.00
Prize Pack Series Cards,6,136.84


In [40]:
df_combined[df_combined["Grandmaster?"] == "no"].groupby("TCGPlayer Category").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
TCGPlayer Category,,
SM - Cosmic Eclipse,5,526.72
Expedition,4,265.76
ME: Ascended Heroes,1,190.00
Plasma Freeze,1,180.00
SV09: Journey Together,1,120.00
Deoxys,5,107.23
Unseen Forces,3,103.14
Sandstorm,4,85.89
Skyridge,2,81.96


In [41]:
df_combined[df_combined["Grandmaster?"] == "yes"].groupby("TCGPlayer Category").agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
).sort_values(by="Total_Price", ascending=False)

,Count,Total_Price
TCGPlayer Category,,
WoTC Promos,1,998.20
Base Set (Shadowless),1,239.99
Alternate Art Promos,1,213.85
SV: Scarlet & Violet Promo Cards,2,180.98
Prize Pack Series Cards,6,136.84
Jumbo Cards,6,120.95
World Championship Decks,16,118.75
Gym Heroes,2,93.74
Miscellaneous Cards & Products,3,91.99


In [42]:
df_combined.agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Price
Count,126.00
Total_Price,4653.43


In [43]:
df_combined[df_combined["Grandmaster?"] == "yes"].agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Price
Count,54.00
Total_Price,2431.08


In [44]:
df_combined[df_combined["Grandmaster?"] == "no"].agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Price
Count,72.00
Total_Price,2222.35


In [45]:
bins = [0, 5, 20, 50, float("inf")]
labels = ["0-5", "5.01-20", "20.01-49.99", "50+"]

df_combined["Price Bucket"] = pd.cut(
    df_combined["Price"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df_combined.groupby(["Grandmaster?", "Price Bucket"], observed=False).agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

Count  Total_Price
Grandmaster? Price Bucket                    
no           0-5              23        61.15
             5.01-20          20       188.55
             20.01-49.99      15       473.52
             50+              14      1499.13
yes          0-5              21        45.13
             5.01-20          17       211.53
             20.01-49.99       6       178.69
             50+              10      1995.73

In [46]:
bins = [0, 5, 20, 50, float("inf")]
labels = ["0-5", "5.01-20", "20.01-49.99", "50+"]

df_combined["Price Bucket"] = pd.cut(
    df_combined["Price"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df_combined.groupby(["Price Bucket"], observed=False).agg(
    Count=("Price", "count"),
    Total_Price=("Price", "sum")
)

,Count,Total_Price
Price Bucket,,
0-5,44,106.28
5.01-20,37,400.08
20.01-49.99,21,652.21
50+,24,3494.86
